<a href="https://colab.research.google.com/github/Rishitha110506/Machine-Learning/blob/main/ML_Lab_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MACHINE LEARNING ASSIGNMENT 07


NAME : KAMPALLI RISHITHA

REG NO : BL.SC.U4AIE24020

SEC : D.Sec

loading data set


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving BERT_Embeddings.xlsx to BERT_Embeddings.xlsx


In [ ]:
import pandas as pd

# Load  dataset
df = pd.read_excel("BERT_Embeddings.xlsx")

# Features (dropping text columns)
X = df.drop(columns=['label','Student','Teacher'])
y = df['label']

(A2)


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier

# Base model
dt = DecisionTreeClassifier(random_state=42)

# Search space
param_dist = {
    'criterion': ['gini','entropy'],
    'max_depth': [None,5,10,20,50],
    'min_samples_split': [2,5,10,20],
    'min_samples_leaf': [1,2,5,10]
}

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=dt,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X,y)

print("Best Parameters:", random_search.best_params_)
print("Best CV Accuracy:", random_search.best_score_)

Best Parameters: {'min_samples_split': 5, 'min_samples_leaf': 10, 'max_depth': 5, 'criterion': 'gini'}
Best CV Accuracy: 0.5031619526732376


In [ ]:
print(df['label'].unique())
print(df['label'].dtype)

[3 2 1]
int64


our dataset’s label column contains discrete integers [1, 2, 3]. That means:

It is not continuous (like scores or measurements), so it’s not regression.

It is not unlabeled (you already have labels), so it’s not clustering.

It is categorical classes (three categories), so your project is a classification problem.

(A3) Our code is a classification pipeline: it takes the embeddings as input features, encodes the target labels `[1,2,3]` into `[0,1,2]` for compatibility, splits the dataset into training and testing sets, and then trains multiple classifiers (DecisionTree, RandomForest, SVM, AdaBoost, XGBoost, CatBoost, Naïve Bayes, and MLP). For each model, it fits on the training data, predicts on both train and test sets, calculates accuracy, and stores the results. Finally, it prints a table comparing train and test accuracy across all models, allowing you to see which algorithm performs best for your dataset.

In [ ]:
!pip install catboost

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder
import pandas as pd

X = df.drop(columns=['label','Student','Teacher'])
y = df['label']

# Encoding labels to start at 0
le = LabelEncoder()
y_encoded = le.fit_transform(y)

#Splitting train and test
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# defining models
models = {
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
    "NaiveBayes": GaussianNB(),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
}

# Training and evaluating
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    results.append({
        "Model": name,
        "Train Accuracy": accuracy_score(y_train, y_pred_train),
        "Test Accuracy": accuracy_score(y_test, y_pred_test)
    })

results_df = pd.DataFrame(results)
print(results_df)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:35:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


          Model  Train Accuracy  Test Accuracy
0  DecisionTree        0.996601       0.405732
1  RandomForest        0.996601       0.541478
2           SVM        0.586480       0.552036
3      AdaBoost        0.585347       0.514329
4       XGBoost        0.996601       0.527903
5      CatBoost        0.996601       0.549020
6    NaiveBayes        0.459592       0.457014
7           MLP        0.995468       0.499246
